# KEEPCODING · AI Engineering · M1 · Agentes con LangChain

## Clase 5 · Lab 5 · Agente con RAG: "La enciclopedia de One Piece"

En los Labs 1-3 viste agentes con tools, grafos y MCP. En el Lab 4 construiste un grafo **Corrective RAG** que se autoverifica sobre la transcripción de un podcast. En este Lab 5 abordamos RAG desde otro ángulo: **RAG como una tool más** dentro de un agente ReAct, junto a una API en vivo y una búsqueda web. Misma idea de fondo (recuperar y generar), formas distintas de combinarla.

RAG es el patrón para que un LLM responda **anclado en tus documentos**: en vez de fiarse solo de lo que el modelo "recuerda", recupera los fragmentos relevantes de un corpus y responde con ellos delante. Sirve para citar fuentes, actualizar el conocimiento sin reentrenar y reducir alucinaciones.

El agente final tendrá **tres herramientas** y decidirá cuál usar en cada pregunta:

- `buscar_en_lore`: RAG sobre una enciclopedia de One Piece escrita en el notebook.
- `buscar_personaje`: la API en vivo de One Piece (recompensas, tripulación).
- `buscar_en_web`: búsqueda web gratuita para datos actuales del mundo real.

### Qué vas a aprender

- Trocear documentos en *chunks* con `RecursiveCharacterTextSplitter`.
- Convertir texto en **embeddings** y guardarlos en un **vector store**.
- Recuperar fragmentos por **similitud semántica**.
- Envolver el retriever como una **tool** y dársela a un agente ReAct.
- Combinar RAG con otras tools y dejar que el agente **elija**.

### Qué necesitas

- Una clave de OpenAI en `OPENAI_API_KEY` (la usan el LLM y los embeddings).
- Este notebook se ejecuta de arriba abajo en **Google Colab** o **Jupyter** local.
- Conexión a internet (embeddings, API de One Piece y búsqueda web).

> Nota: ejecuta las celdas en orden. Cada celda depende de las anteriores.

### Por qué importa

RAG no siempre es el grafo entero: muchas veces es **una tool más** que el agente decide usar junto a una API y una búsqueda web. Saber qué forma encaja en cada caso es lo que separa una demo de un sistema.

### Conecta con

- **Slides M1** · RAG dentro de un agente.
- **Lab 4** · Corrective RAG (el contraste: grafo autoverificante vs RAG como tool).
- **Clase 4 / M3** · el pipeline RAG completo.
- **M5** · cómo se registran las tools en un harness.

### Stack

`langchain>=1.0` · `langchain-openai` · `ddgs` · `gpt-5-mini`.

### Hardware

Cualquier CPU; Colab Free sobra (no usa GPU, todo es API).


## Guion del laboratorio

1. **Setup**: instalar dependencias y cargar la API key.
2. **¿Qué es RAG?**: el patrón recuperar-y-generar en cuatro pasos.
3. **El corpus**: una enciclopedia de One Piece escrita en el notebook.
4. **Chunking**: trocear los documentos en fragmentos.
5. **Embeddings y vector store**: indexar los chunks para buscarlos por significado.
6. **Probar la recuperación**: `similarity_search` sin agente todavía.
7. **LLM desnudo vs RAG**: por qué anclar la respuesta en documentos.
8. **Tres tools**: el retriever, la API de One Piece y la búsqueda web.
9. **El agente RAG**: `create_agent` con las tres herramientas.
10. **Demo**: el agente elige la tool correcta para cada pregunta.
11. **Recap y retos**.

## 1. Setup

### 1.1. Instalar dependencias

- `langchain` y `langchain-openai`: el framework, el chat model y los embeddings.
- `langgraph`: el runtime sobre el que corre el agente de `create_agent`.
- `ddgs`: cliente de DuckDuckGo para la búsqueda web **gratuita y sin API key**.

> En Colab esto tarda unos segundos. La opción `-q` reduce el ruido en la salida.

In [1]:
# Instalación de dependencias (ejecutar una sola vez por sesión)
%pip install -q "langchain>=1.0" langchain-openai langchain-text-splitters "langgraph>=1.0" ddgs numpy

### 1.2. Cargar la API key de OpenAI

Esta celda intenta tres fuentes en orden: la variable de entorno, los Secrets de
Google Colab y, por último, un cuadro de entrada seguro con `getpass`. Así el
notebook funciona igual en Colab y en Jupyter local.

La `OPENAI_API_KEY` se usa **dos veces** en este lab: para el LLM del agente y
para generar los *embeddings* del vector store.

In [2]:
import os
from getpass import getpass


def cargar_api_key() -> None:
    """Asegura que OPENAI_API_KEY esté disponible en el entorno."""
    if os.environ.get("OPENAI_API_KEY"):
        print("OPENAI_API_KEY encontrada en las variables de entorno.")
        return

    # Intento 2: secretos de Google Colab.
    try:
        from google.colab import userdata

        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        print("OPENAI_API_KEY cargada desde los secretos de Colab.")
        return
    except Exception:
        pass

    # Intento 3: entrada manual segura.
    os.environ["OPENAI_API_KEY"] = getpass("Introduce tu OPENAI_API_KEY: ")
    print("OPENAI_API_KEY cargada desde la entrada manual.")


cargar_api_key()

OPENAI_API_KEY encontrada en las variables de entorno.


## 2. ¿Qué es RAG?

**RAG** (*Retrieval-Augmented Generation*) es un patrón en cuatro pasos:

1. **Indexar** (una vez): se trocea un corpus de documentos, se convierte cada
   trozo en un **vector** (embedding) y se guarda en un **vector store**.
2. **Recuperar**: ante una pregunta, se buscan en el vector store los trozos
   **más parecidos en significado** (no por palabras exactas).
3. **Aumentar**: esos trozos se añaden al prompt del LLM como contexto.
4. **Generar**: el LLM responde usando ese contexto.

La idea clave: el modelo no inventa desde su memoria, **responde leyendo tus
documentos**. Eso da respuestas citables, actualizables sin reentrenar y con
menos alucinaciones.

En este lab montamos los pasos 1-2 a mano y luego envolvemos la recuperación en
una **tool**, para que un agente la use cuando lo crea necesario.

## 3. El corpus: la enciclopedia de One Piece

Un sistema RAG necesita documentos. Aquí escribimos seis fichas de lore de One
Piece directamente en el notebook: así el lab es 100% reproducible, sin depender
de ficheros externos ni de la red para el corpus.

Cada documento tiene un **título** (que usaremos como *fuente* citable) y un
**texto**. En un caso real, estos documentos vendrían de tus PDFs, tu wiki
interna o tu base de conocimiento.

In [3]:
# La enciclopedia: seis fichas de lore. En un proyecto real vendrían de
# PDFs, una wiki o una base de datos documental.
DOCUMENTOS = [
    {
        "titulo": "La geografía del mundo de One Piece",
        "texto": (
            "El mundo de One Piece está dominado por dos accidentes geográficos. "
            "La Red Line es un continente-anillo que rodea el planeta de polo a "
            "polo. La Grand Line es un océano que cruza la Red Line en "
            "perpendicular y donde ocurre casi toda la aventura. La Grand Line "
            "está flanqueada a ambos lados por el Calm Belt, dos franjas de mar "
            "sin viento ni corrientes y plagadas de Reyes del Mar, que la hacen "
            "casi inaccesible. Estas dos líneas dividen el mar en cuatro: el "
            "North Blue, el East Blue, el West Blue y el South Blue. La Grand "
            "Line se recorre en dos mitades: la primera, más suave, se conoce "
            "como Paradise; la segunda, brutal, es el New World. Para entrar se "
            "sube por la Reverse Mountain, una montaña por la que el agua fluye "
            "hacia arriba. La navegación normal no sirve: hace falta un Log "
            "Pose, una brújula que se calibra con el campo magnético de las islas."
        ),
    },
    {
        "titulo": "Las Frutas del Diablo",
        "texto": (
            "Las Frutas del Diablo son frutas misteriosas que conceden un poder "
            "sobrehumano a quien se las come. Hay tres tipos. Las Paramecia dan "
            "una habilidad sobre el propio cuerpo o el entorno, como volverse de "
            "goma o generar ondas de choque. Las Zoan permiten transformarse en "
            "un animal y en formas intermedias; existen variantes míticas y "
            "ancestrales, más raras y poderosas. Las Logia permiten convertir el "
            "cuerpo en un elemento natural, como fuego, humo o luz, y son las "
            "más codiciadas. El precio es severo: quien come una Fruta del "
            "Diablo pierde para siempre la capacidad de nadar y el mar le "
            "debilita. Solo existe una fruta de cada tipo a la vez en el mundo; "
            "si su portador muere, el poder reaparece en otra fruta. Un usuario "
            "experto puede 'despertar' su fruta y alcanzar un nivel de poder "
            "muy superior."
        ),
    },
    {
        "titulo": "El Haki",
        "texto": (
            "El Haki es una energía espiritual que todo ser vivo posee, aunque "
            "muy pocos aprenden a usarla. Hay tres tipos. El Haki de Observación "
            "(Kenbunshoku) permite percibir la presencia, la emoción y la "
            "intención de los demás, e incluso anticipar sus movimientos unos "
            "instantes antes de que ocurran. El Haki de Armadura (Busoshoku) "
            "crea una armadura invisible que refuerza ataque y defensa, y es la "
            "única forma de golpear a un usuario de fruta Logia. El Haki del "
            "Conquistador (Haoshoku) es el más raro: solo lo tiene una persona "
            "entre un millón y permite imponer la voluntad propia sobre los "
            "demás, llegando a dejar inconscientes a los rivales más débiles. "
            "Los tres tipos se entrenan y refinan; los luchadores de elite del "
            "New World combinan Armadura y Observación en cada combate."
        ),
    },
    {
        "titulo": "El Siglo Vacío y los Poneglyphs",
        "texto": (
            "El Siglo Vacío es un periodo de cien años de la historia sobre el "
            "que el Gobierno Mundial prohíbe investigar. Se cree que en esa "
            "época existió un reino poderoso que fue destruido por una alianza "
            "de veinte reinos, la misma que después fundó el Gobierno Mundial. "
            "La única memoria de aquel tiempo está grabada en los Poneglyphs, "
            "bloques de piedra indestructibles escritos en una lengua antigua "
            "que casi nadie sabe leer. Estudiar los Poneglyphs se castiga con la "
            "muerte. Existen Poneglyphs especiales, los Road Poneglyphs: hay "
            "cuatro y, leídos juntos, señalan la posición de Laugh Tale, la isla "
            "final donde espera el tesoro conocido como One Piece. La capacidad "
            "de leer estas piedras es lo que hace tan valiosa, y tan perseguida, "
            "a la arqueóloga de los Sombrero de Paja."
        ),
    },
    {
        "titulo": "El sistema de poder del mundo",
        "texto": (
            "El equilibrio de fuerzas del mundo se sostiene sobre varios "
            "bloques. El Gobierno Mundial agrupa a la mayoría de los países y su "
            "brazo armado es la Marina, dirigida por almirantes de enorme poder. "
            "Por encima opera Cipher Pol, una red de agencias de inteligencia, y "
            "en la cúspide del Gobierno están los Cinco Ancianos. Frente a "
            "ellos, los Cuatro Emperadores o Yonko son los piratas más "
            "poderosos, que se reparten el dominio del New World. Durante mucho "
            "tiempo existió un tercer pilar, los Siete Guerreros del Mar o "
            "Shichibukai: piratas a los que el Gobierno perdonaba sus crímenes a "
            "cambio de servir como fuerza de choque; ese sistema fue finalmente "
            "abolido. La recompensa o bounty que el Gobierno pone sobre la "
            "cabeza de un pirata es la medida oficial de lo peligroso que se le "
            "considera."
        ),
    },
    {
        "titulo": "La tripulación del Sombrero de Paja",
        "texto": (
            "Los Piratas del Sombrero de Paja son la tripulación protagonista, "
            "fundada por su capitán en el East Blue. Su bandera, su Jolly Roger, "
            "es una calavera con un sombrero de paja. Cada miembro se une por un "
            "sueño propio: el capitán quiere ser el Rey de los Piratas "
            "encontrando el One Piece; el espadachín, ser el mejor del mundo; la "
            "navegante, dibujar el mapa completo del mundo; el cocinero, hallar "
            "el All Blue, un mar legendario; el francotirador, volverse un "
            "valiente guerrero del mar; el médico, curar cualquier enfermedad; "
            "la arqueóloga, descubrir la verdad del Siglo Vacío. La tripulación "
            "navega en el Thousand Sunny, un barco de madera rarísima. Su fuerza "
            "no es solo el poder individual, sino la confianza entre sus miembros."
        ),
    },
]

print(f"Enciclopedia cargada: {len(DOCUMENTOS)} documentos.")
for d in DOCUMENTOS:
    print(f"  - {d['titulo']} ({len(d['texto'])} caracteres)")

Enciclopedia cargada: 6 documentos.
  - La geografía del mundo de One Piece (853 caracteres)
  - Las Frutas del Diablo (799 caracteres)
  - El Haki (773 caracteres)
  - El Siglo Vacío y los Poneglyphs (766 caracteres)
  - El sistema de poder del mundo (787 caracteres)
  - La tripulación del Sombrero de Paja (727 caracteres)


## 4. Trocear los documentos (chunking)

No metemos los documentos enteros en el vector store. Los partimos en **chunks**
(fragmentos) por dos razones:

- **Precisión**: la búsqueda devuelve el fragmento concreto que responde a la
  pregunta, no un documento entero donde la respuesta es una sola línea.
- **Límite de contexto**: al LLM le pasamos solo los trozos relevantes, no toda
  la enciclopedia.

Usamos `RecursiveCharacterTextSplitter`: parte por párrafos y frases intentando
no cortar a mitad de una idea. `chunk_size` es el tamaño máximo de cada trozo y
`chunk_overlap` es cuánto se solapan dos trozos seguidos (un colchón para no
perder contexto en los bordes).

Cada chunk guarda en sus **metadatos** el título de su documento de origen: eso
nos permitirá **citar la fuente** más adelante.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Convertimos cada ficha en un Document de LangChain con su fuente en metadata.
documentos_lc = [
    Document(page_content=d["texto"], metadata={"fuente": d["titulo"]})
    for d in DOCUMENTOS
]

# El splitter trocea respetando párrafos y frases siempre que puede.
splitter = RecursiveCharacterTextSplitter(chunk_size=350, chunk_overlap=60)
chunks = splitter.split_documents(documentos_lc)

print(f"{len(documentos_lc)} documentos -> {len(chunks)} chunks\n")
print("Ejemplo de chunk:")
print(f"  fuente: {chunks[0].metadata['fuente']}")
print(f"  texto : {chunks[0].page_content[:160]}...")

6 documentos -> 18 chunks

Ejemplo de chunk:
  fuente: La geografía del mundo de One Piece
  texto : El mundo de One Piece está dominado por dos accidentes geográficos. La Red Line es un continente-anillo que rodea el planeta de polo a polo. La Grand Line es un...


## 5. Embeddings y vector store

Un **embedding** es la conversión de un texto en un vector de números que
captura su **significado**. Dos textos que hablan de lo mismo tienen vectores
cercanos, aunque no compartan ni una palabra.

El **vector store** es la base de datos que guarda esos vectores y sabe buscar
los más cercanos a una consulta. Usamos:

- `OpenAIEmbeddings` con el modelo `text-embedding-3-small`: barato y rápido.
- `InMemoryVectorStore`: un vector store que vive en memoria. Perfecto para un
  lab; en producción usarías Chroma, FAISS, pgvector o Pinecone.

Al indexar, LangChain llama a la API de embeddings una vez por chunk y guarda
cada vector junto a su texto y sus metadatos.

In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

# El modelo de embeddings: convierte texto en vectores de significado.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Creamos el vector store e indexamos los chunks (una llamada de embedding
# por chunk). A partir de aquí ya podemos buscar por significado.
vector_store = InMemoryVectorStore.from_documents(chunks, embeddings)

print(f"Vector store listo con {len(chunks)} chunks indexados.")

Vector store listo con 18 chunks indexados.


## 6. Probar la recuperación

Antes de meter nada en un agente, comprobamos el corazón de RAG: la búsqueda por
similitud. `similarity_search` recibe una consulta, la convierte en un vector y
devuelve los `k` chunks más cercanos en significado.

Fíjate en que la consulta no comparte las palabras exactas con los documentos:
la búsqueda funciona por **significado**, no por coincidencia literal.

In [6]:
consulta = "¿Qué debilidad tiene quien gana poderes al comer algo extraño?"

resultados = vector_store.similarity_search(consulta, k=3)

print(f"Consulta: {consulta}\n")
print(f"Chunks recuperados ({len(resultados)}):\n")
for i, doc in enumerate(resultados, start=1):
    print(f"{i}. [fuente: {doc.metadata['fuente']}]")
    print(f"   {doc.page_content[:170]}...\n")

Consulta: ¿Qué debilidad tiene quien gana poderes al comer algo extraño?

Chunks recuperados (3):

1. [fuente: Las Frutas del Diablo]
   Las Frutas del Diablo son frutas misteriosas que conceden un poder sobrehumano a quien se las come. Hay tres tipos. Las Paramecia dan una habilidad sobre el propio cuerpo...

2. [fuente: Las Frutas del Diablo]
   el mar le debilita. Solo existe una fruta de cada tipo a la vez en el mundo; si su portador muere, el poder reaparece en otra fruta. Un usuario experto puede 'despertar' ...

3. [fuente: Las Frutas del Diablo]
   intermedias; existen variantes míticas y ancestrales, más raras y poderosas. Las Logia permiten convertir el cuerpo en un elemento natural, como fuego, humo o luz, y son ...



## 7. Por qué anclar la respuesta en documentos

Quizá pienses: "este modelo ya sabe de One Piece, ¿para qué RAG?". RAG no es solo
para lo que el modelo ignora. Sirve para algo más importante: **anclar la
respuesta en una fuente concreta que tú controlas**.

Con RAG, la respuesta:

- **Se puede citar**: sabes de qué documento salió.
- **Se puede actualizar**: cambias el documento y la respuesta cambia, sin
  reentrenar el modelo.
- **Es más fiable**: el modelo trabaja con el texto delante, no de memoria.

Abajo comparamos el LLM solo y el mismo LLM con el chunk recuperado como
contexto. La diferencia no es tanto *acertar* como **saber de dónde sale la
respuesta**.

In [7]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5-mini")
pregunta = "¿Qué son los Road Poneglyphs y adónde llevan?"

# A) El LLM solo, de memoria.
solo = llm.invoke(pregunta).content

# B) El LLM con el contexto recuperado del vector store.
contexto = vector_store.similarity_search(pregunta, k=2)
texto_contexto = "\n\n".join(
    f"[{d.metadata['fuente']}] {d.page_content}" for d in contexto
)
prompt_rag = (
    "Responde la pregunta usando SOLO este contexto. "
    "Cita la fuente entre corchetes.\n\n"
    f"Contexto:\n{texto_contexto}\n\nPregunta: {pregunta}"
)
con_rag = llm.invoke(prompt_rag).content

print("PREGUNTA:", pregunta)
print("=" * 65)
print("A) LLM solo (de memoria):")
print(solo)
print("-" * 65)
print("B) LLM + RAG (con el contexto recuperado):")
print(con_rag)

PREGUNTA: ¿Qué son los Road Poneglyphs y adónde llevan?
A) LLM solo (de memoria):
Los Road Poneglyphs (ロードポーネグリフ) son una clase especial de Poneglyphs en One Piece. Puntos clave:

- Qué son: son piedras antiguas similares a los Poneglyphs normales pero de color y función especiales; están hechas para marcar la ruta al último gran misterio del mundo.  
- Cuántos hay: solo existen cuatro Road Poneglyphs.  
- Para qué sirven: cada uno contiene coordenadas/indicios que, combinados, señalan la ubicación exacta de Laugh Tale (antes llamado Raftel), la isla final del Grand Line. Es necesario encontrar y leer los cuatro para poder navegar hasta Laugh Tale.  
- Quién puede leerlos: solo personas que sepan leer Poneglyphs (como Nico Robin) pueden descifrar su contenido.  
- Importancia/posesión: por su valor estratégico, suelen estar en manos de fuerzas poderosas (Yonkō, gobiernos, etc.) y son objetos codiciados tanto por la Marina como por piratas que buscan el One Piece.  
- Origen: al igual q

## 8. Las tres herramientas del agente

Ya tenemos RAG funcionando. Ahora lo convertimos en una **tool** y se la damos a
un agente, junto a otras dos. El agente decidirá cuál usar en cada pregunta.

| Tool | Para qué | De dónde saca los datos |
|------|----------|-------------------------|
| `buscar_en_lore` | Lore y worldbuilding | El vector store (RAG) |
| `buscar_personaje` | Datos de un personaje | La API en vivo de One Piece |
| `buscar_en_web` | Información actual del mundo real | Búsqueda web (DuckDuckGo) |

### 8.1. `buscar_en_lore`: RAG como una tool

Una tool de RAG no es más que una función que hace `similarity_search` y
devuelve los chunks encontrados. El agente la llama, lee el resultado y construye
su respuesta con él.

In [8]:
from langchain_core.tools import tool


@tool
def buscar_en_lore(consulta: str) -> str:
    """Busca en la enciclopedia de lore de One Piece: geografía, Frutas del
    Diablo, Haki, el Siglo Vacío, el sistema de poder y la tripulación.

    Úsala para preguntas sobre el mundo y las reglas de One Piece, NO para
    datos concretos de un personaje. Devuelve fragmentos con su fuente.
    """
    encontrados = vector_store.similarity_search(consulta, k=3)
    if not encontrados:
        return "No hay nada en la enciclopedia sobre esa consulta."
    bloques = []
    for d in encontrados:
        bloques.append(f"[fuente: {d.metadata['fuente']}]\n{d.page_content}")
    return "\n\n".join(bloques)


# Prueba directa de la tool.
print(buscar_en_lore.invoke({"consulta": "tipos de Haki"})[:320], "...")

[fuente: El Haki]
El Haki es una energía espiritual que todo ser vivo posee, aunque muy pocos aprenden a usarla. Hay tres tipos. El Haki de Observación (Kenbunshoku) permite percibir la presencia, la emoción y la intención de los demás, e incluso anticipar sus movimientos unos instantes antes de que ocurran. El Haki de ...


### 8.2. `buscar_personaje`: la API en vivo de One Piece

La misma tool de los Labs 1-3: consulta la API pública de One Piece y, si falla,
cae a un diccionario local. El matcher de nombres tolera puntos, acentos y
nombres completos (da igual "Luffy" que "Monkey D. Luffy").

In [9]:
import urllib.request
import json
import unicodedata

# Cache en memoria: la lista completa de personajes se descarga una sola vez.
_OP_CACHE = []

# Plan B: recompensas canónicas post-Wano de los 10 Sombrero de Paja.
_OP_FALLBACK = {
    "luffy": ("Monkey D Luffy", "3.000.000.000"),
    "zoro": ("Roronoa Zoro", "1.111.000.000"),
    "nami": ("Nami", "366.000.000"),
    "sanji": ("Sanji", "1.032.000.000"),
    "usopp": ("Usopp", "500.000.000"),
    "chopper": ("Tony-Tony Chopper", "1.000"),
    "robin": ("Nico Robin", "930.000.000"),
    "franky": ("Franky", "394.000.000"),
    "brook": ("Brook", "383.000.000"),
    "jinbe": ("Jinbe", "1.100.000.000"),
}


def _normalizar(texto: str) -> str:
    """Minúsculas, sin acentos ni puntuación, espacios colapsados."""
    t = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode()
    t = "".join(c if c.isalnum() else " " for c in t.lower())
    return " ".join(t.split())


def _coincide(consulta: str, nombre: str) -> bool:
    """True si 'consulta' identifica a 'nombre'. Tolera puntos, acentos y orden."""
    q, n = _normalizar(consulta), _normalizar(nombre)
    if not q or not n:
        return False
    if q == n or q in n or n in q:
        return True
    palabras = [w for w in q.split() if len(w) >= 3]
    return bool(palabras) and all(w in n.split() for w in palabras)


def _cargar_personajes_op():
    """Descarga (una sola vez) la lista completa de personajes de One Piece."""
    global _OP_CACHE
    if _OP_CACHE:
        return _OP_CACHE
    url = "https://api.api-onepiece.com/v2/characters/en"
    req = urllib.request.Request(url, headers={"User-Agent": "keepcoding-ai-lab/1.0"})
    with urllib.request.urlopen(req, timeout=25) as r:
        _OP_CACHE = json.load(r)
    return _OP_CACHE


@tool
def buscar_personaje(nombre: str) -> str:
    """Busca la recompensa (bounty) de un personaje de One Piece por su nombre.

    Intenta primero la API en vivo de One Piece y, si falla, usa una base de
    datos local con las recompensas canónicas de la tripulación de Sombrero
    de Paja. Devuelve un texto con el nombre y la recompensa en berries.
    """
    # Intento 1: API en vivo (lista completa cacheada, se filtra en Python).
    try:
        for c in _cargar_personajes_op():
            if _coincide(nombre, c.get("name") or ""):
                return (
                    f"{c['name']}: recompensa "
                    f"{c.get('bounty') or 'desconocida'} berries (One Piece API)."
                )
        return f"No se encontró a '{nombre}' en la One Piece API."
    except Exception as e:
        # Cualquier fallo de red o de datos nos lleva al plan B.
        for k, (n, b) in _OP_FALLBACK.items():
            if _coincide(nombre, k) or _coincide(nombre, n):
                return f"{n}: recompensa {b} berries (fallback local)."
        return f"No se pudo consultar la API ni el fallback para '{nombre}': {e}"


# Prueba directa de la tool.
print(buscar_personaje.invoke({"nombre": "Zoro"}))

Roronoa Zoro: recompensa 1.101.000.000 berries (One Piece API).


### 8.3. `buscar_en_web`: búsqueda web gratuita

Para datos del mundo real (noticias, estrenos, personas reales) que no están ni
en la enciclopedia ni en la API, el agente busca en la web. Usamos **DuckDuckGo**
a través de la librería `ddgs`: gratis y **sin API key**.

Como toda tool robusta, captura los errores (DuckDuckGo a veces limita las
peticiones) y devuelve un mensaje útil en vez de romper el agente.

In [10]:
from ddgs import DDGS


@tool
def buscar_en_web(consulta: str) -> str:
    """Busca información actual en la web con DuckDuckGo.

    Úsala para datos del mundo real (noticias, estrenos, fechas, personas
    reales) que no estén en la enciclopedia de lore ni sean datos de un
    personaje de ficción. Devuelve títulos y extractos con su URL.
    """
    try:
        with DDGS() as ddg:
            resultados = list(ddg.text(consulta, max_results=4))
    except Exception as error:
        return (
            f"No se pudo buscar en la web ('{consulta}'): {error}. "
            "DuckDuckGo puede estar limitando peticiones; inténtalo de nuevo."
        )
    if not resultados:
        return f"Sin resultados web para '{consulta}'."
    lineas = []
    for r in resultados:
        lineas.append(
            f"- {r.get('title', '')}\n  {r.get('body', '')}\n  ({r.get('href', '')})"
        )
    return "\n".join(lineas)


# Prueba directa de la tool.
print(buscar_en_web.invoke({"consulta": "One Piece manga"})[:260], "...")

- One Piece manga (English version)
  One Piece (stylized in all caps) is a Japanese manga series written and illustrated by Eiichiro Oda. It follows the adventures of Monkey D. Luffy and his crew, the Straw Hats, as he searches for the legendary treasure know ...


## 9. El agente RAG

Montamos el agente con `create_agent` y las tres tools. La pieza que hace
que el agente **elija bien** es el *system prompt*: le explicamos para qué sirve
cada herramienta. El agente, en su bucle ReAct, leerá la pregunta, decidirá qué
tool encaja, la llamará y construirá la respuesta con el resultado.

In [11]:
from langchain.agents import create_agent  # LangChain v1.0+ · sustituye al deprecado create_react_agent

PROMPT_SISTEMA = (
    "Eres un experto en One Piece con tres herramientas:\n"
    "- 'buscar_en_lore': la enciclopedia de lore (geografía, Frutas del Diablo, "
    "Haki, Siglo Vacío, sistema de poder, la tripulación). Para preguntas de "
    "mundo y reglas.\n"
    "- 'buscar_personaje': datos concretos de un personaje (recompensa, "
    "tripulación) desde la API en vivo.\n"
    "- 'buscar_en_web': información actual del mundo real (noticias, estrenos, "
    "personas reales).\n"
    "Elige la herramienta según la pregunta. Si usas 'buscar_en_lore', cita "
    "entre corchetes la fuente del fragmento. Responde en español, claro y "
    "conciso."
)

agente = create_agent(
    model=ChatOpenAI(model="gpt-5-mini"),
    tools=[buscar_en_lore, buscar_personaje, buscar_en_web],
    system_prompt=PROMPT_SISTEMA,
)

print("Agente RAG construido con 3 tools:")
print("  buscar_en_lore, buscar_personaje, buscar_en_web")

Agente RAG construido con 3 tools:
  buscar_en_lore, buscar_personaje, buscar_en_web


## 10. Demo: el agente elige la herramienta

Le hacemos tres preguntas pensadas para disparar **una tool distinta cada una**.
La función de ayuda recorre el stream del agente y etiqueta cada paso: qué tool
llama (*Action*), qué le devuelve (*Observation*) y la respuesta final.

In [12]:
from langchain_core.messages import AIMessage, ToolMessage


def preguntar_al_agente(pregunta: str) -> None:
    """Ejecuta el agente en streaming y muestra el bucle ReAct paso a paso."""
    print(f"PREGUNTA: {pregunta}\n")
    entrada = {"messages": [{"role": "user", "content": pregunta}]}
    for fragmento in agente.stream(entrada, stream_mode="values"):
        ultimo = fragmento["messages"][-1]
        if isinstance(ultimo, AIMessage):
            for llamada in ultimo.tool_calls:
                print(f"  [Action]      {llamada['name']}({llamada['args']})")
            if not ultimo.tool_calls and ultimo.content:
                print(f"\n  [Respuesta]   {ultimo.content}")
        elif isinstance(ultimo, ToolMessage):
            extracto = " ".join((ultimo.content or "").split())
            print(f"  [Observation] {ultimo.name}: {extracto[:120]}...")
    print("\n" + "-" * 65)

### 10.1. Pregunta de lore -> `buscar_en_lore`

Una pregunta sobre el mundo de One Piece. El agente debería recuperar de la
enciclopedia y citar la fuente.

In [13]:
preguntar_al_agente("¿Qué fue el Siglo Vacío y por qué el Gobierno lo oculta?")

PREGUNTA: ¿Qué fue el Siglo Vacío y por qué el Gobierno lo oculta?



  [Action]      buscar_en_lore({'consulta': '¿Qué fue el Siglo Vacío en One Piece? ¿Por qué el Gobierno Mundial oculta la información sobre el Siglo Vacío? Explica causas y evidencias conocidas (Poneglyphs, Rio Poneglyph, armas antiguas, reino antiguo, D. y Historia) y cita fuentes de la enciclopedia.'})


  [Observation] buscar_en_lore: [fuente: El Siglo Vacío y los Poneglyphs] El Siglo Vacío es un periodo de cien años de la historia sobre el que el Gobie...


  [Action]      buscar_en_lore({'consulta': 'Poneglyphs, Rio Poneglyph, armas antiguas (Pluton, Poseidon, Uranus), Reino Antiguo, clan D., y por qué el Gobierno Mundial oculta el Siglo Vacío. Citas de la enciclopedia.'})


  [Observation] buscar_en_lore: [fuente: El Siglo Vacío y los Poneglyphs] El Siglo Vacío es un periodo de cien años de la historia sobre el que el Gobie...



  [Respuesta]   Resumen breve
- El Siglo Vacío es un periodo de cien años de la historia mundial cuyos registros fueron borrados y cuya investigación está prohibida por el Gobierno Mundial. Se supone que entonces existió un Reino Antiguo poderoso que fue derrotado por una alianza de veinte reinos; esos vencedores fundaron después el Gobierno Mundial. [fuente: El Siglo Vacío y los Poneglyphs]

Qué sabemos y cuáles son las evidencias
- Poneglyphs: La única memoria escrita del Siglo Vacío son los Poneglyphs, enormes bloques de piedra indestructibles escritos en una lengua antigua que casi nadie puede leer. El estudio o la búsqueda de esos textos está castigada con la muerte. [fuente: El Siglo Vacío y los Poneglyphs]
- Road Poneglyphs y Laugh Tale: Hay cuatro Road Poneglyphs especiales; leídos juntos señalan la ubicación de Laugh Tale, la isla que guarda el One Piece y la verdad final de la historia. [fuente: El Siglo Vacío y los Poneglyphs]
- Río (Rio) Poneglyph / Historia Verdadera: En 

### 10.2. Dato de un personaje -> `buscar_personaje`

Una pregunta de datos concretos. El agente debería ir a la API en vivo, no a la
enciclopedia.

In [14]:
preguntar_al_agente("¿Cuál es la recompensa actual de Monkey D. Luffy?")

PREGUNTA: ¿Cuál es la recompensa actual de Monkey D. Luffy?



  [Action]      buscar_personaje({'nombre': 'Monkey D. Luffy'})
  [Observation] buscar_personaje: Monkey D Luffy: recompensa 3.000.000.000 berries (One Piece API)....



  [Respuesta]   La recompensa actual de Monkey D. Luffy es de 3.000.000.000 berries (3 mil millones de berries), según la One Piece API.

-----------------------------------------------------------------


### 10.3. Dato del mundo real -> `buscar_en_web`

Una pregunta que no está ni en la enciclopedia ni en la API: algo actual del
mundo real. El agente debería buscar en la web.

In [15]:
preguntar_al_agente(
    "¿Quién es el actor que interpreta a Luffy en la serie de One Piece de Netflix?"
)

PREGUNTA: ¿Quién es el actor que interpreta a Luffy en la serie de One Piece de Netflix?



  [Action]      buscar_en_web({'consulta': 'actor que interpreta a Luffy en la serie de Netflix One Piece Iñaki Godoy'})


  [Observation] buscar_en_web: - Iñaki Godoy - Wikipedia, la enciclopedia libre En 2021, Godoy fue anunciado como el actor que personificaría a Monkey ...



  [Respuesta]   El actor es Iñaki Godoy, quien interpreta a Monkey D. Luffy en la serie live-action de One Piece de Netflix.

-----------------------------------------------------------------


## 11. Recap y retos

### Lo que has construido

Has montado un sistema **RAG** completo y se lo has dado a un agente:

| Pieza | Qué hace | En este lab |
|---|---|---|
| Chunking | Trocea los documentos | `RecursiveCharacterTextSplitter` |
| Embeddings | Texto a vectores de significado | `OpenAIEmbeddings` |
| Vector store | Guarda y busca por similitud | `InMemoryVectorStore` |
| Retrieval | Recupera los chunks relevantes | `similarity_search` |
| Tool de RAG | Envuelve la recuperación | `@tool buscar_en_lore` |
| Agente | Elige entre RAG, API y web | `create_agent` |

### Ideas clave

- RAG = recuperar trozos relevantes y generar la respuesta con ellos delante.
- La búsqueda es **semántica**: por significado, no por palabras exactas.
- Guardar la **fuente** en los metadatos permite citar de dónde sale cada dato.
- RAG no es magia: es una tool más. El agente decide cuándo usarla.
- RAG actualiza el conocimiento **sin reentrenar**: cambias el documento y ya.

### Retos para practicar

1. **Amplía la enciclopedia**: añade un documento nuevo (por ejemplo, sobre las
   islas del New World), reindexa y pregunta por él.
2. **Sube el `k`**: cambia `similarity_search(..., k=3)` a `k=5` y observa si el
   agente cita más fuentes.
3. **Juega con el chunking**: prueba `chunk_size=150` y `chunk_size=600`. ¿Cómo
   cambia la precisión de la recuperación?
4. **Pregunta mixta**: pide "compara la recompensa de Zoro con lo que dice la
   enciclopedia sobre el Haki" y observa cómo el agente encadena dos tools.
5. **Memoria**: añade un `checkpointer=MemorySaver()` (como en el Lab 2) para que
   el agente recuerde la conversación entre preguntas.